# Unit 8+9: Multiclass classification, BERT sentiment, and BERTopic

Three workflows in one notebook:
1. **Multiclass classification** — extend the binary MLP from Unit 7 to predict *one of several* classes (penguin species).
2. **BERT sentiment** — compare a classical dictionary approach (VADER) with a pretrained transformer.
3. **BERTopic** — topic modeling with BERT embeddings.

At the very end, we bring back **LDA** as a classical baseline to contrast with BERTopic.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import confusion_matrix
from palmerpenguins import load_penguins

import matplotlib.pyplot as plt
import seaborn as sns

# Sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

# Topic modeling
import nltk
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

import gensim
from gensim import corpora
from gensim.models.ldamodel import LdaModel
from bertopic import BERTopic

import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

---
# Part 1: Multiclass classification (Penguins)

Predict penguin species (Adélie / Chinstrap / Gentoo) from body measurements. Same PyTorch skeleton as Unit 7, but the output layer now has one neuron per class.

In [ ]:
peng = load_penguins()

### Pre-processing data
* Using `pd.factorize()`, transform the categorical variables into numbers.
* Impute missing values with the column mean.
* Then, using `train_test_split`, generate an 80/20 split.

In [ ]:
# your code

In [ ]:
# your code here

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
# Define model
class penguinNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(penguinNet, self).__init__()
        #your architecture

    def forward(self, x):
        # your forward solution
        # note the last pass does not need to go through activation / non-linear

**Note on the loss:** `nn.CrossEntropyLoss` internally applies `log_softmax` to your raw outputs and then computes negative log-likelihood. That's why your `forward` should return raw logits — *don't* apply softmax yourself, or you'll apply it twice.

In [ ]:
# Model parameters / You can vary this as you wish
input_size = 7
hidden_size = 4
output_size = 3
model = penguinNet(input_size, hidden_size, output_size)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Training loop
num_epochs = 1000
for epoch in range(num_epochs):
    # your training loop

#### Run this to evaluate your model!

In [ ]:
with torch.no_grad():
    # Forward pass on the test set
    outputs = model(X_test_tensor)

    # Get predicted classes
    _, ypred = torch.max(outputs, 1)

    # Calculate accuracy
    total = y_test_tensor.size(0)
    correct = (ypred == y_test_tensor).sum().item()
    accuracy = correct / total * 100
    print(f"Test Accuracy: {accuracy:.2f}%")

### Generate a confusion matrix of the classifications

**Hint:** use `sklearn.metrics.confusion_matrix(y_true, y_pred)` to get the matrix, then plot it with `sns.heatmap(cm, annot=True, fmt='d')`.

In [ ]:
# your code

---
# Part 2: BERT sentiment

Compare a **dictionary-based** approach (VADER) with a **pretrained transformer** (BERT). We'll use 500 documents from the 20 Newsgroups corpus.

In [ ]:
# Load sample dataset
newsgroups_data = fetch_20newsgroups(remove=('headers', 'footers', 'quotes'))
documents = newsgroups_data.data[:500]  # Limit to 500 documents for faster computation

df = pd.DataFrame(documents, columns=["text"])

### VADER
This is the traditional off-the-shelf approach.

In [ ]:
# Initialize VADER Sentiment Analyzer
vader_analyzer = SentimentIntensityAnalyzer()
df["vaderSent"] = df.text.apply(lambda x: vader_analyzer.polarity_scores(x))

### BERT

**Why 500 characters?** BERT models have a **512-token input limit** — anything longer gets dropped. Truncating to 500 characters is a quick way to stay under that.

In [ ]:
df["trunc_text"] = df.text.apply(lambda x: x[:500])

`pipeline('sentiment-analysis')` defaults to **`distilbert-base-uncased-finetuned-sst-2-english`** — a smaller distilled BERT fine-tuned on the Stanford Sentiment Treebank. Output is `{'label': 'POSITIVE'/'NEGATIVE', 'score': float}`.

In [ ]:
bert_classifier = pipeline('sentiment-analysis')
bert_scores = bert_classifier(df.trunc_text.tolist())

In [ ]:
df["bertSent"] = bert_scores

### Check the correlation between the two sentiment pipelines

**Hint:** the two outputs aren't directly comparable. Pull a single signed score from each:

- VADER: `vaderSent['compound']` is already in [−1, +1].
- BERT: combine `label` and `score` — `score` if positive, `−score` if negative.

Then `.corr()` on the two columns.

In [ ]:
# your code

### Find examples where the two sentiment classifications are opposing. Why might this be the case?

**Hint:** filter for rows where one score is strongly positive and the other is strongly negative — e.g., `(vader_score > 0.5) & (bert_score < -0.5)` (or the reverse).

In [ ]:
# your code

---
# Part 3: BERTopic

BERTopic represents documents as **BERT embeddings**, reduces them with UMAP, and clusters them with HDBSCAN. Each cluster becomes a topic. It works on raw text — no manual preprocessing needed.

In [1]:
documents = newsgroups_data.data[:2000]

NameError: name 'newsgroups_data' is not defined

In [ ]:
%%time
topic_model = BERTopic(min_topic_size=5)
topics, probs = topic_model.fit_transform(documents)

In [ ]:
topic_model.get_topic_info()

### Under the hood

BERTopic does three things:

1. **Embed** each document with a pretrained sentence transformer (BERT-based)
2. **Reduce** dimensionality with UMAP (typically to ~5 dims)
3. **Cluster** the reduced embeddings with HDBSCAN — each cluster becomes a topic

That's also why it doesn't need preprocessing: the embeddings encode meaning even with stopwords.

In [ ]:
topic_model.visualize_topics()

### Semantic search over topics

Because BERTopic represents topics as embeddings, you can ask *"which topics are most about X?"* — even if X never appears in the corpus.

In [ ]:
# Returns the top-5 topics whose embeddings are closest to the query
topic_model.find_topics("technology", top_n=5)

---
# Part 4: LDA (classical baseline)

**Latent Dirichlet Allocation** was the standard topic-modeling approach before neural methods. It models each document as a mixture over topics, and each topic as a distribution over words. It needs *much* more preprocessing (stopword removal, stemming) because it operates on word counts, not semantic embeddings.

In [ ]:
list_stopwords = stopwords.words("english")
porter = PorterStemmer()

def process_step(one_str):
    nostop_listing = [word for word in wordpunct_tokenize(one_str)
                      if word not in list_stopwords]
    clean_listing = [porter.stem(word) for word in nostop_listing
                    if word.isalpha()
                    and len(word) > 3]
    clean_listing_str = " ".join(clean_listing)
    return(clean_listing_str)

df["text_proc"] = df.text.apply(process_step)

In [ ]:
# Preprocess data
documents_clean = [gensim.utils.simple_preprocess(doc) for doc in df.text_proc]
dictionary = corpora.Dictionary(documents_clean)
corpus = [dictionary.doc2bow(doc) for doc in documents_clean]

In [ ]:
lda_model = LdaModel(corpus=corpus, num_topics=10, id2word=dictionary, random_state=42)

In [ ]:
topics = lda_model.print_topics(num_words = 10)

for topic in topics:
    print(topic)

In [ ]:
pyLDAvis.enable_notebook()

### visualize
lda_display = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(lda_display)